# NLS metadata — joined dataframe

Replaces the six separate NLS spreadsheets with one joined dataframe to explore the "what counts as a programme" question.

This notebook builds on top of the
analysis work already committed in
[`../batch_analysis.ipynb`](../batch_analysis.ipynb) (duration histogram,
year distribution, type-frequency chart, `sample-11.csv` generation). See
that notebook for those.

**Requires**: RDS mount (CREATE) + KCL credentials. Run `../copy-metadata.bash`
first, or manually copy the eleven metadata CSVs into
`<repo-root>/data/input/NLS/batch2/NLS Metadata/`.


In [43]:
import re
from pathlib import Path
import pandas as pd

## 1. Config / load

In [44]:
METADATA_DIR = Path("/mnt/rds_issa/data/input/NLS/batch2/NLS Metadata") # local RDS mount point, change if needed

REQUIRED_FILES = [
    "FILMS.csv",
    "Clips_Table.csv",
    "GENRE.csv",
    "SERIES.csv",
    "PERSONALITIES.csv",
    "SPECIFIC_CATEGORIES.csv",
    "FILM_GENRE_ASSIGN.csv",
    "FILM_SERIES_ASSIGN.csv",
    "FILM_PERSONALITY_ASSIGN.csv",
    "FILM_CATEGORY_ASSIGN.csv",
]

missing = [f for f in REQUIRED_FILES if not (METADATA_DIR / f).exists()]
if not METADATA_DIR.exists() or missing:
    raise FileNotFoundError(
        f"NLS metadata not found in {METADATA_DIR.resolve()}.\n"
        f"Missing: {missing or '(folder itself is missing)'}\n"
        "Run ../copy-metadata.bash first (requires the RDS mount + KCL "
        "credentials), or copy the files there manually."
    )

def read_nls_csv(name):
    return pd.read_csv(METADATA_DIR / name, encoding="cp1252")


## 2. Join into one dataframe

`FILMS` is the spine (`ref no`). `Clips_Table` adds `DODfilenameprefix` — the
field that maps to real archive filenames (e.g. `139329389`). The four
`FILM_*_ASSIGN` junction tables are resolved to names via their lookup
tables and aggregated into **lists per film** — a film can have several
genres/series/personalities/categories, and we don't want to fan out rows
over that.

`type` is normalised on the way in: stripped and split on `\n`, `,`, or `;` —
there are ~90 raw string variants from inconsistent delimiters and trailing
spaces (e.g. `'tv news'` vs `'tv news '` vs `'tv sport\ntv news'`).

Two things about the `FILM_*_ASSIGN` tables that aren't obvious from the
column names alone:

- **The film-side key is composite, not a plain `ref no`.** It's a
  zero-padded number with an optional letter prefix/suffix and a trailing
  assignment-sequence number, e.g. `'1111_1'` → `1111`, `'T0845_1'` → `845`,
  `'2852A_1'` → `2852`. `extract_ref_no()` below parses this (confirmed
  against `cKey1Title`, the assign tables' cached film-title column — e.g.
  `'1111_1'`'s `cKey1Title` is "ROBOT THREE", matching `FILMS`' `ref no`
  1111). A handful of rows per table are genuinely malformed (nulls, a
  leaked `'uniqueID'` header value, stray whitespace) and are dropped.
- **These tables are archive-wide**, covering NLS's entire collection —
  thousands of films — not just our ~400-film batch. So most rows in each
  assign table resolve to films outside `df_films`; only a fraction of our
  batch has any genre/series/personality/category assignment at all
  (Cell 5 reports the exact batch-coverage rate per table).

In [45]:
df_films = read_nls_csv("FILMS.csv")
df_clips = read_nls_csv("Clips_Table.csv")

df_genre = read_nls_csv("GENRE.csv")
df_series = read_nls_csv("SERIES.csv")
df_personalities = read_nls_csv("PERSONALITIES.csv")
df_categories = read_nls_csv("SPECIFIC_CATEGORIES.csv")

df_film_genre = read_nls_csv("FILM_GENRE_ASSIGN.csv")
df_film_series = read_nls_csv("FILM_SERIES_ASSIGN.csv")
df_film_personality = read_nls_csv("FILM_PERSONALITY_ASSIGN.csv")
df_film_category = read_nls_csv("FILM_CATEGORY_ASSIGN.csv")


In [46]:
def normalise_type(raw):
    """Strip whitespace, split on newline/comma/semicolon, drop empties."""
    if pd.isna(raw):
        return []
    parts = re.split(r"[\n,;]+", str(raw))
    return [p.strip() for p in parts if p.strip()]


df_films["type_list"] = df_films["type"].apply(normalise_type)
df_films["type_list"].head()


0               [advertising]
1             [instructional]
2               [educational]
3               [educational]
4    [sponsored, documentary]
Name: type_list, dtype: object

In [47]:
def extract_ref_no(composite_key):
    """The film-side key on every FILM_*_ASSIGN table is a composite:
    an optional letter prefix + a zero-padded ref no + an optional letter
    suffix + an assignment-sequence suffix, e.g. '1111_1' -> 1111,
    'T0845_1' -> 845, '2852A_1' -> 2852. Confirmed by cross-referencing
    cKey1Title (e.g. '1111_1' resolves to "ROBOT THREE", matching FILMS'
    ref no 1111). Returns None for the handful of genuinely malformed keys
    (stray nulls, a leaked 'uniqueID' header value, stray whitespace/control
    characters) — dropped rather than crashing the parse.
    """
    m = re.match(r"^[A-Za-z]*(\d+)[A-Za-z]*_", str(composite_key))
    return int(m.group(1)) if m else None


def resolve_multivalued(assign_df, film_key_col, lookup_df, lookup_key, lookup_name_col, assign_lookup_key=None):
    """Join an assign (junction) table to its lookup table, then aggregate
    the resolved names into a list per film ref no — no row fan-out on df_films.

    NOTE: these assign tables are archive-wide (thousands of films across
    all of NLS's collection), not scoped to our ~400-film batch — expect
    most of each table's rows to resolve to films outside df_films.
    """
    assign_df = assign_df.copy()
    assign_df["ref no"] = assign_df[film_key_col].apply(extract_ref_no)
    assign_df = assign_df.dropna(subset=["ref no"])
    assign_lookup_key = assign_lookup_key or lookup_key
    resolved = assign_df.merge(
        lookup_df[[lookup_key, lookup_name_col]],
        left_on=assign_lookup_key, right_on=lookup_key, how="left",
    )
    return resolved.groupby("ref no")[lookup_name_col].apply(list)


genres_by_film = resolve_multivalued(
    df_film_genre, "film ID", df_genre, "id", "genre term", assign_lookup_key="genre ID"
)
series_by_film = resolve_multivalued(
    df_film_series, "filmID", df_series, "seriesid", "seriesName", assign_lookup_key="seriesID"
)
personalities_by_film = resolve_multivalued(
    df_film_personality, "foreignKey1", df_personalities, "personalityID", "full_name", assign_lookup_key="foreignKey2"
)
categories_by_film = resolve_multivalued(
    df_film_category, "foreignKey1", df_categories, "id", "specific_category_name", assign_lookup_key="foreignKey2"
)


In [48]:
df = df_films.merge(
    df_clips[["ref no", "DODfilenameprefix"]], on="ref no", how="left"
)

for col_name, series in [
    ("genres", genres_by_film),
    ("series", series_by_film),
    ("personalities", personalities_by_film),
    ("categories", categories_by_film),
]:
    df[col_name] = df["ref no"].map(series)
    df[col_name] = df[col_name].apply(lambda v: v if isinstance(v, list) else [])

df.sample(5)


,uniqueID,ref no,title,date,dateOfReleaseYYYY,productionStartYYYY,type,copyrightClearance,Non-Fiction/ Fiction,colour,...,original format,TV Episode Title,TV Episode No.,Tx Date,type_list,DODfilenameprefix,genres,series,personalities,categories
155,7761_1,7761,"YOUNG PROFESSIONALS, the",1979c,1979,NaN,amateur comedy,NO,Fiction,col,...,super 8mm,NaN,NaN,NaN,[amateur comedy],139493753,"[amateur, comedy]",[],[],"[Lanarkshire, Leisure and Recreation, Transpor..."
78,2682_1,2682,BENNO SCHOTZ: Sculptor and Modeller,1973,1973,NaN,educational,NO,non-fiction,col,...,16mm,NaN,NaN,NaN,[educational],102738972,[educational],[],"[McConnell, Edward ‘Eddie’]","[Arts and Crafts, Education, Glasgow, Science ..."
56,2223_1,2223,"BORDERS: Where Scotland and England Meet, the",1970,1970,NaN,"sponsored, documentary",NO,non-fiction,col,...,35mm,NaN,NaN,NaN,"[sponsored, documentary]",78297376,"[documentary, sponsored]",[],"[Films of Scotland Committee, Grigor, Murray]","[nan, nan, nan, nan, Berwickshire, nan, Border..."
21,0935_1,935,CRIEFF HIGHLAND GATHERING,1950,1950,NaN,amateur,NO,non-fiction,col,...,16mm,NaN,NaN,NaN,[amateur],75247337,[amateur],[],[],"[nan, nan, Sporting Activities, Perth, Celts a..."
120,4742_1,4742,I WILL BE,1978,1978,NaN,educational,YES,non-fiction,col,...,16mm,NaN,NaN,NaN,[educational],189392856,[educational],[Teenage Talk-In],[],"[Emotions, Attitudes and Behaviour, Home Life,..."


## 3. Testing "programme = tape" vs "programme = shotlist item"

Flag whether each row's `shotlist` matches the discrete-timecode pattern
(one line per item: `HH:MM:SS<TAB>HH:MM:SS`). Crosstabbed against `type`,
this should reproduce: **200/200 `tv news` rows match, 0/200 others match** —
meaning NLS's own cataloguing already distinguishes multi-programme tapes
(Grampian) from single-programme tapes (even if they are described as compilations).

In [49]:
TIMECODE_PATTERN = re.compile(r"\d{2}:\d{2}:\d{2}\t\d{2}:\d{2}:\d{2}")

df["shotlist_is_timecoded"] = df["shotlist"].apply(
    lambda s: bool(TIMECODE_PATTERN.search(str(s))) if pd.notna(s) else False
)

df["is_tv_news"] = df["type_list"].apply(lambda types: "tv news" in types)

pd.crosstab(df["is_tv_news"], df["shotlist_is_timecoded"])


shotlist_is_timecoded,False,True
is_tv_news,,
False,200,0
True,0,200


## 4. "compilation" language check

Search `synopsis` / `additionalAndContextualInformation` for the word
"compilation", crosstabbed against the timecode-pattern flag from secton 3.
Should reproduce: **172 hits, only 3 outside the timecoded (Grampian) group**
— refs 1493, 7340, 7726. My view is that NLS's catalogue uses "compilation"
loosely (a tape with several disparate scenes), not necessarily "several
separate programmes" — confirm with them with these 3 examples.

In [50]:
text_cols = [c for c in ["synopsis", "additionalAndContextualInformation"] if c in df.columns]
search_text = df[text_cols].fillna("").agg(" ".join, axis=1)

df["mentions_compilation"] = search_text.str.contains("compilation", case=False)

print(pd.crosstab(df["mentions_compilation"], df["shotlist_is_timecoded"]))

exceptions = df.loc[df["mentions_compilation"] & ~df["shotlist_is_timecoded"], "ref no"]
print("\nCompilation mentions outside the timecoded group (ref no):", list(exceptions))


shotlist_is_timecoded  False  True 
mentions_compilation               
False                    197     31
True                       3    169

Compilation mentions outside the timecoded group (ref no): [1493, 7340, 7726]


## 5. Additional descriptive stats

Only what isn't already covered in `batch_analysis.ipynb` (duration
histogram, year distribution, type-frequency chart live there). Non-null
rates for TV Episode Title / Tx Date / Producer (tv) / Certificate are
expected to be near-zero. This to me signals that NLS's catalogue carries no
structured item-level metadata beyond `shotlist`

In [51]:
pd.crosstab(df["sound"], df["colour"])


colour,bw,bwcol,col,sepia
sound,,,,
mute,2,0,0,0
silent,31,4,31,0
sound,31,2,298,1


In [52]:
df["decade"] = (df["dateOfReleaseYYYY"] // 10 * 10).astype("Int64")
df["decade"].value_counts().sort_index()


decade
1900      3
1920      3
1930     17
1940     22
1950     49
1960     45
1970     40
1980    208
1990      7
2000      2
2010      1
2020      3
Name: count, dtype: Int64

In [53]:
for col in ["genres", "series", "personalities", "categories"]:
    has_any = df[col].apply(len).gt(0)
    print(f"{col}: {has_any.sum()}/{len(df)} films have at least one")


genres: 179/400 films have at least one
series: 22/400 films have at least one
personalities: 100/400 films have at least one
categories: 179/400 films have at least one


In [54]:
sparse_cols = ["TV Episode Title", "Tx Date", "Producer (tv)", "Certificate"]
present = [c for c in sparse_cols if c in df.columns]
df[present].notna().mean().rename("non-null rate")


TV Episode Title    0.0025
Tx Date             0.0000
Producer (tv)       0.0050
Certificate         0.0050
Name: non-null rate, dtype: float64

## 6. NLS review worklist

`sample-11.csv` (generated by `batch_analysis.ipynb`, kept as-is) already
has a draft `progs` count per tape (Geoffroy's human turth annotations). Join it against the full table here to
see which of the 16 already have a `progs` count vs which are still at
`progs=0` — I believe these are part of the sample but haven't been human-annotated so that can be a working list for the NLS review task.

In [55]:
sample = pd.read_csv("../sample-11.csv")

worklist = sample.merge(
    # "ref no" and "title" already live in sample-11.csv itself — pull only
    # the new columns from df, to avoid a merge collision (pandas would
    # otherwise silently suffix both to "ref no_x"/"ref no_y").
    df[["DODfilenameprefix", "type_list", "shotlist_is_timecoded"]],
    on="DODfilenameprefix",
    how="left",
)

print(f"{(worklist['progs'] > 0).sum()}/{len(worklist)} already have a progs count")
worklist.loc[worklist["progs"] == 0, ["DODfilenameprefix", "ref no", "title"]]

12/16 already have a progs count


,DODfilenameprefix,ref no,title
9,140179431,10807,[GRAMPIAN TELEVISION NEWS TAPE L0103]
10,140180335,10777,[GRAMPIAN TELEVISION NEWS TAPE L0144]
12,144133880,12043,[GRAMPIAN TELEVISION NEWS TAPE L0858]
13,231298910,9071,[MAC MOVIES ARCHIVE COMPILATION 3]


## Findings from the joined NLS metadata analysis

### The two collections

The 400 files shared with us split cleanly into two batches (collecctions to keep with FrameSense nomenclature) by `type` — 200 `tv news`
(Grampian) and 200 everything else (non-Grampian). And the two batches behave
differently as far we have observed:

- **Grampian tapes likely bundle multiple items per file**, and their
  shotlist's discrete timecode structure (100% consistent) is a
  strong candidate source for programme boundaries and counts. But we
  haven't confirmed that **one shotlist line = one broadcast programme** —
  it could just as easily be a rush/raw-take or any different unit,(Geoffroy's
  original worry). We need to manually check if this can be reliably taken as a per-programme segmentation.
- **Non-Grampian files are likely single-programme files (with some identified exceptions)**
  At least 4 known exceptions (refs 1493, 7340, 7726, 9071 — see below) use
  "compilation" language while looking structurally identical to genuine
  single-programme films. The shotlist-format signal alone can't tell them
  apart from the other 196, but Paul has seen them and his account is a single programme (or fragments thereof).

So both sides face the same underlying question — "what does one
catalogued unit actually correspond to on tape". We need a working definiton, from NLS for Grampian and spot-checking exceptions for
non-Grampian.

### What the metadata confirms

1. **Shotlist encoding is a clean, 100% bimodal split.**
   Every one of the 200 `tv news` (Grampian) rows has a discrete, timecoded
   shotlist (`HH:MM:SS<TAB>HH:MM:SS` per line); *none* of the other 200 rows
   do — no exceptions. NLS's cataloguing already encodes a programme/segment distinction
   structurally. Is it safe to asume that the non-Grampian group is one programme per file?

2. **"Compilation" language tracks the Grampian group, but isn't a reliable
   proxy for it.** 172/400 synopses mention "compilation": 169 inside the
   timecoded (Grampian) group, only 3 outside it (refs **1493, 7340, 7726**).
   But the reverse gap is there too — **31 of the 200 Grampian tapes never
   use the word "compilation" at all**, despite being structurally identical
   multi-programme tapes. So the word "compilation" is neither necessary nor sufficient — the
   shotlist timecode structure is the more reliable signal, the language is fuzzy.

3. **This check has a blind spot: it only scanned `synopsis` and
   `additionalAndContextualInformation`, not `title`.** Ref 9071
   ("MAC MOVIES ARCHIVE COMPILATION 3") has "compilation" in its *title*
   only, so it doesn't show up in the count above at all. The true
   non-Grampian "compilation"-language footprint is probably larger than 3, but still likely just an artefact of language, i.e. some programmes will be called "Compilations" but can be reasonably treated as single programmes.

4. **No structured programme-level metadata exists beyond the shotlist field.**
   TV Episode Title, Tx Date, Producer (tv), Certificate are all <1% non-null
   across all 400 files. The only segment-level structure info is `shotlist` text, but we'd need to check the time-codes are reliable.

5. **Genre/category assignment covers ~45% of the two colelctions (overlapping, genres 179/400; categories 179/400), series covers only 5.5% (22/400); personalities 25% (100/400).**
   These lookup tables are archive-wide, not scoped to our two
   batches, so the low coverage only represents how thin that
   metadata is for this particular set of files. Something to keep in mind for anything we want to target in the schema.

6. **The batch is overwhelmingly 1980s material** (208/400 films, mostly
   the Grampian tapes) with a long thin front tail to 1900s–1920s. This is likely relevant for the
   programme boundary detection task across H2022 material.

### Answers to some of our questions

- **"How do they use/encode the shotlist column?"** — we can now say
  precisely *what* the two uses are (see #1) and that NLS's own data
  supports the idea that "programme = file" for single-programme catalogue entries vs.
  "programme = shotlist item" for Grampian tapes. We still don't know **how or by whom** this was
  encoded — this is something we can ask NLS directly.
- **"Single programme called "compilation" vs. several programmes compiled in one file"** 
  Concretely: ask NLS about refs 1493, 7340, 7726
  (compilation language, single-programme structure) and 9071 (compilation
  *title*, not yet reviewed at all) — these four can help us calrify if"compilation" in the text should be taken to mean mean one programme or several? in either collection.
- **"What's the desired output?"**
  Still open, still resting on the UFA-card signal from earlier and on NLS's own answer.

### Still needs a human eye to check the actual video/shotlist

1. **Refs 1493, 7340 (85060728), 7726 (106033956)** — "compilation" in the
   synopsis, but structurally single-programme. Paul's manual review of
   7726 already found it isn't fully silent (a later voice-over comes in
   at 17:47). All three need an actual watch to align the definition to archive expectation.
2. **Ref 9071 (231298910, "MAC MOVIES ARCHIVE COMPILATION 3")** — title
   says compilation, `progs` is still 0 (hasn't been annotated), and it's outside
   the Grampian group. Best single next annotation target — directly tests
   the ambiguity of how to define programme vs segment.
3. **Refs 10807 (140179431), 10777 (140180335), 12043 (144133880)** — the
   three remaining Grampian tapes still at `progs=0`. Since the timecode
   pattern appears reliable across all 200 Grampian tapes, these are
   very likely a straightforward count-the-shotlist-items task.
4. **Spot-check a couple of the 31 "compilation"-silent Grampian tapes** —
   confirm the presence of the shotlist-timecode is a reliable signal of multi-programme.


## 8. Dashboard data export

Writes a small **aggregated** summary — file counts, coverage rates, known
vs pending states — to `../metadata_hierarchy_data.json`, which
`../metadata_hierarchy.html` (the public workshop dashboard, styled like
`experiments/qwen3x/results.html`) fetches and renders. This is the one
intentional exception to "never export to a tracked path" — it's a
deliberate, small, public-facing aggregate, not the raw dataframe, and it's
meant to be committed and reused across workshops the same way
`evaluations.csv` already is.

Re-run this cell any time the upstream data changes — the dashboard reads
whatever's in the JSON, no other changes needed.

In [56]:
import json

# Count discrete timecoded shotlist items per row (not just flag whether any
# exist) — turns the Grampian "programme" band from a single "unknown"
# placeholder into a real per-tape candidate count.
df["shotlist_item_count"] = df["shotlist"].apply(
    lambda s: len(TIMECODE_PATTERN.findall(str(s))) if pd.notna(s) else 0
)

# "Verified" means a segments_true/*.json file exists AND actually has
# segments in it — not just that the file is present. A few files (e.g.
# 231298910) have a placeholder json with zero segments recorded, same as
# batch_analysis.ipynb's count_segments(): that's "not yet annotated", the
# same thing progs==0 means in sample-11.csv, not "verified".
segments_true_dir = Path("../segments_true")

def count_true_segments(dod_prefix):
    if pd.isna(dod_prefix):
        return 0
    p = segments_true_dir / f"{int(dod_prefix)}.32.json"
    if not p.exists():
        return 0
    return len(json.loads(p.read_text()))

df["verified_segment_count"] = df["DODfilenameprefix"].apply(count_true_segments)
df["has_verified_segments"] = df["verified_segment_count"] > 0

# Slice into collections AFTER both columns above exist on df — a slice
# taken earlier wouldn't pick up columns added to df afterwards, and
# grampian["has_verified_segments"] etc. below would KeyError.
grampian = df[df["is_tv_news"]]
non_grampian = df[~df["is_tv_news"]]

exceptions_refs = df.loc[
    df["mentions_compilation"] & ~df["shotlist_is_timecoded"], "ref no"
].tolist()

# Per-file detail for the 16-film working sample (Cell 6), so the dashboard
# can show where it actually falls across the two collections — not just
# the aggregate "12/16" count. `shotlist_is_timecoded` (joined in from df at
# Cell 6) is the same 100%-reliable Grampian/non-Grampian split used
# everywhere else in this notebook.
sample_items = [
    {
        "dod": str(row["DODfilenameprefix"]),
        "ref_no": int(row["ref no"]),
        "title": row["title"],
        "is_grampian": bool(row["shotlist_is_timecoded"]),
        "progs": int(row["progs"]),
    }
    for _, row in worklist.iterrows()
]

dashboard_data = {
    "generated_at": pd.Timestamp.now().isoformat(timespec="minutes"),
    "archive": "NLS",
    "collections": [
        {
            "id": "grampian",
            "name": "Grampian (tv news)",
            "file_count": int(len(grampian)),
            "programme": {
                "status": "estimated_pending_verification",
                "note": (
                    "Count of discrete timecoded shotlist lines per tape — a "
                    "strong candidate for programme count, not yet confirmed "
                    "that one line always equals one broadcast programme."
                ),
                "estimated_count_distribution": {
                    str(k): int(v)
                    for k, v in grampian["shotlist_item_count"].value_counts().sort_index().items()
                },
                "verified_file_count": int(grampian["has_verified_segments"].sum()),
            },
            "segment": {
                "status": "unknown",
                "note": "No segment-level ground truth exists yet for Grampian items.",
            },
        },
        {
            "id": "non_grampian",
            "name": "non-Grampian (single programme)",
            "file_count": int(len(non_grampian)),
            "programme": {
                "status": "assumed_one_per_file",
                "note": "Default assumption; known exceptions listed below.",
                "known_exception_ref_no": exceptions_refs,
                "verified_file_count": int(non_grampian["has_verified_segments"].sum()),
            },
            "segment": {
                "status": "shotlist_prose_unvalidated",
                "note": (
                    "Shotlist gives prose shot descriptions — usable as "
                    "inner-segmentation ground truth, not yet validated."
                ),
            },
        },
    ],
    "coverage": {
        col: {
            "films_with_at_least_one": int(df[col].apply(len).gt(0).sum()),
            "total_films": int(len(df)),
        }
        for col in ["genres", "series", "personalities", "categories"]
    },
    "sparse_fields_non_null_rate": df[present].notna().mean().round(4).to_dict(),
    "sample_worklist": {
        "total": int(len(worklist)),
        "with_progs": int((worklist["progs"] > 0).sum()),
        "pending_dod": worklist.loc[worklist["progs"] == 0, "DODfilenameprefix"].astype(str).tolist(),
        "items": sample_items,
    },
    "compilation_language_check": {
        "total_films_mentioning_compilation": int(df["mentions_compilation"].sum()),
        "inside_grampian_group": int((df["mentions_compilation"] & df["shotlist_is_timecoded"]).sum()),
        "outside_grampian_group": int((df["mentions_compilation"] & ~df["shotlist_is_timecoded"]).sum()),
        "grampian_files_not_using_the_word": int((grampian["shotlist_is_timecoded"] & ~grampian["mentions_compilation"]).sum()),
        "known_blind_spot": (
            "Search only scanned synopsis/additionalAndContextualInformation, "
            "not title — e.g. ref 9071 'MAC MOVIES ARCHIVE COMPILATION 3' is missed."
        ),
    },
}

OUT_PATH = Path("../metadata_hierarchy_data.json")
OUT_PATH.write_text(json.dumps(dashboard_data, indent=2))
print(f"Wrote {OUT_PATH.resolve()}")


Wrote /home/daniel/Projects/issa/workshops/ws1/metadata_hierarchy_data.json
